In [2]:
import matplotlib.pyplot as plt
import json
import glob
import os
import pandas as pd
import seaborn as sns

In [3]:
LANGUAGES = ["PB NATIVO", "ESPANHOL", "FRANCÊS", "GALEGO", "INGLÊS", "MANDARIM", "RUSSO", "ÁRABE"]
N_LANGUAGES = len(LANGUAGES) 


TASKS = ["CAPACITAR", "COMPARTILHAR", "EXPLICAR", "EXPLORAR", "FAZER", "RECOMENDAR", "RECRIAR", "RELATAR"]
N_TASKS = len(TASKS)

typeLM = {
    "causalLM": {
        "files": glob.glob(f"./causalLM/*.json"),
        "metrics": ["avg_tokens_per_word", "avg_ppl", "avg_bpb"]
    },
    "maskedLM": {
        "files": glob.glob(f"./maskedLM/*.json"),
        "metrics": ["avg_tokens_per_word", "avg_pseudo_ppl", "avg_bpb"]
    },
}

typeLM

{'causalLM': {'files': ['./causalLM/Tucano.json',
   './causalLM/Sabia.json',
   './causalLM/Qwen.json',
   './causalLM/Gemma.json',
   './causalLM/BODE.json',
   './causalLM/Cabrita.json',
   './causalLM/Gervasio.json'],
  'metrics': ['avg_tokens_per_word', 'avg_ppl', 'avg_bpb']},
 'maskedLM': {'files': ['./maskedLM/Serafim335m.json',
   './maskedLM/BERTimbauLarge.json',
   './maskedLM/Albertina100m.json',
   './maskedLM/BERTimbau.json',
   './maskedLM/Albertina900m.json'],
  'metrics': ['avg_tokens_per_word', 'avg_pseudo_ppl', 'avg_bpb']}}

In [12]:
for metric_name in ("ppl", "avg_bpb"):

    for lm_type, lm_info in typeLM.items():
        files_data = {}

        if metric_name == "ppl":
            metric_json = "avg_ppl"
            metric_key = "Perplexity"

            if "avg_ppl" not in lm_info["metrics"]:
                metric_json = "avg_pseudo_ppl"
                metric_key = "Pseudo Perplexity"

        else:
            metric_json = "avg_bpb"
            metric_key = "BPB"

        print(metric_json, metric_key)

        for file_path in lm_info["files"]:

            model_name = os.path.basename(file_path).replace(".json", "")

            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            files_data[model_name] = data

        for task in TASKS:
            chart_data = {
                "Tokens/Word": [],
                metric_key: [],
                "Model": [],
                "Language": []
            }

            for model, results in files_data.items():
                PT_BR = results[task]["PB NATIVO"][metric_json]

                for language in LANGUAGES:
                    if language == "PB NATIVO":
                        continue

                    value = results[task][language][metric_json]
                    delta = value / PT_BR - 1

                    tokens_per_word = results[task][language]["avg_tokens_per_word"]

                    chart_data["Tokens/Word"].append(tokens_per_word)
                    chart_data[metric_key].append(delta)
                    chart_data["Model"].append(model)
                    chart_data["Language"].append(language)

            df = pd.DataFrame(chart_data)

            plt.figure(figsize=(10, 6))

            sns.scatterplot(
                data=df,
                x="Tokens/Word",
                y=metric_key,
                hue="Language",
                style="Model",
                s=80
            )

            plt.legend(
                bbox_to_anchor=(1.05, 1),
                loc="upper left"
            )

            plt.title(f"{lm_type} - {task}")
            plt.tight_layout()

            plt.savefig(
                f"./charts/{lm_type}_{task}_{metric_key}.pdf",
                format="pdf",
                bbox_inches="tight"
            )

            plt.close()

avg_ppl Perplexity
avg_pseudo_ppl Pseudo Perplexity
avg_bpb BPB
avg_bpb BPB
